In [1]:
import numpy as np
import pandas as pd

# Load the merged DOAS dataset
df = pd.read_csv('DOAS_urdaneta_2014_2021_combined.csv')

# Clean negative sensor error codes (e.g. -999) to NaN
pollutants = ['so2', 'nox', 'o3', 'pm2.5', 'pm10', 'co']
for col in pollutants:
  df[col] = pd.to_numeric(df[col], errors='coerce')
  df.loc[df[col] < 0, col] = np.nan

# Define EPA / NAAQIS Breakpoints for PM2.5 and PM10
pm25_bp = [
    (0.0, 12.0, 0, 50),
    (12.1, 35.4, 51, 100),
    (35.5, 55.4, 101, 150),
    (55.5, 150.4, 151, 200),
    (150.5, 250.4, 201, 300),
    (250.5, 500.4, 301, 500),
]

pm10_bp = [
    (0, 54, 0, 50),
    (55, 154, 51, 100),
    (155, 254, 101, 150),
    (255, 354, 151, 200),
    (355, 424, 201, 300),
    (425, 604, 301, 500),
]


def calculate_aqi(conc, breakpoints):
  if pd.isna(conc) or conc < 0:
    return np.nan
  for c_low, c_high, i_low, i_high in breakpoints:
    if c_low <= conc <= c_high:
      return ((i_high - i_low) / (c_high - c_low)) * (conc - c_low) + i_low
  return 500.0 if conc > breakpoints[-1][1] else np.nan


# Compute sub-indices and overall AQI
df['aqi_pm25'] = df['pm2.5'].apply(lambda c: calculate_aqi(c, pm25_bp))
df['aqi_pm10'] = df['pm10'].apply(lambda c: calculate_aqi(c, pm10_bp))
df['aqi'] = df[['aqi_pm25', 'aqi_pm10']].max(axis=1)

# Save as the exact cleaned file in your VS Code workspace
df.to_csv('cleaned_DOAS_urdaneta_2014_2021.csv', index=False)
print("Saved! Columns are now:", df.columns.tolist())

Saved! Columns are now: ['date', 'year', 'month', 'day', 'so2', 'nox', 'o3', 'pm2.5', 'pm10', 'co', 'aqi_pm25', 'aqi_pm10', 'aqi']
